# Phase 1 — Data Collection & Preprocessing

Join the Fiserv Small Business Index to macro data, align monthly, clean,
and build growth rates and lags.

Output: `data/processed/integrated_monthly.csv`

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.width", 180)
pd.set_option("display.max_columns", 40)

DATA = Path("../data") if Path("../data").exists() else Path("data")
(DATA / "processed").mkdir(exist_ok=True)

## 1. Fiserv

Two exports covering the same panel — one nominal, one deflated. Headers
come out with inconsistent spacing, so clean them on the way in.

In [ ]:
def load(filename):
    df = pd.read_csv(DATA / filename)
    df.columns = (df.columns.str.replace(r"\s+", " ", regex=True)
                  .str.replace("Real ", "").str.replace(" - ", "_")
                  .str.replace(" ", "_").str.lower())
    df["date"] = pd.to_datetime(df["period"], format="%Y%m%d")
    return df.rename(columns={"sector_name": "sector", "sub-sector_name": "subsector"})


nom = load("Data_Nominal.csv")
real = load("Data_InflationAdjusted.csv")

print(nom.columns.tolist())
print(f"\n{len(nom):,} rows, {nom.date.nunique()} months, {nom.geo.nunique()} geographies")

Check the two files line up before joining anything. The transaction index
appears in both and should match exactly — deflation scales sales, not
transaction counts.

In [ ]:
assert len(nom) == len(real)
assert (nom.transactional_index_sa == real.transactional_index_sa).all()
print("ok")

## 2. National series

We want the `US / ALL / ALL` headline. Average ticket is sales divided by
transactions — dollars per basket.

Use the nominal file for this: Fiserv builds the real index by deflating
with CPI, so anything from the nominal/real ratio partly reconstructs CPI
instead of predicting it.

In [ ]:
us = nom.query("geo == 'US' and sector == 'ALL' and subsector == 'ALL'").set_index("date")

fiserv = pd.DataFrame({
    "sales_sa": us.sales_index_sa,
    "txn_sa": us.transactional_index_sa,
    "sales_nsa": us.sales_index_nsa,
    "txn_nsa": us.transactional_index_nsa,
}).sort_index()

fiserv["ticket_sa"] = fiserv.sales_sa / fiserv.txn_sa * 100
fiserv["ticket_nsa"] = fiserv.sales_nsa / fiserv.txn_nsa * 100

fiserv = fiserv.add_prefix("fsbi_")
fiserv.tail(3).round(1)

In [ ]:
# Cross-check against Fiserv's published June 2026 release: index 145,
# sales +2.4% YoY, average ticket +3.7% YoY.
jun = fiserv.loc["2026-06-01"]
yoy = lambda s: np.log(s).diff(12).loc["2026-06-01"] * 100

print(f"index       {jun.fsbi_sales_sa:6.1f}   published 145")
print(f"sales YoY   {yoy(fiserv.fsbi_sales_sa):6.1f}%  published 2.4%")
print(f"ticket YoY  {yoy(fiserv.fsbi_ticket_sa):6.1f}%  published 3.7%")

Small gaps are revision — the June release used the July vintage.

## 3. Macro

FRED carries the BLS price indices and Census retail sales as well, so one
source covers everything. Cached after the first pull.

In [ ]:
SERIES = {
    "CPIAUCSL": "cpi_sa", "CPIAUCNS": "cpi_nsa", "CPILFESL": "core_cpi_sa",
    "PCEPI": "pce_price", "PCEPILFE": "core_pce_price",

    "CUSR0000SEFV": "cpi_food_away", "CUSR0000SETB01": "cpi_gasoline",
    "CUSR0000SAF11": "cpi_food_home", "CUSR0000SAH1": "cpi_shelter",
    "CPIAPPSL": "cpi_apparel", "CUSR0000SETA02": "cpi_used_cars",

    "RSAFS": "retail_sa", "RSAFSNA": "retail_nsa", "RSFSXMV": "retail_ex_auto",
    "PCE": "pce", "PCEC96": "real_pce", "PCEDG": "pce_durables",
    "PCEND": "pce_nondurables", "PCES": "pce_services",
    "DSPIC96": "real_income", "PSAVERT": "savings_rate",

    "UNRATE": "unemployment", "PAYEMS": "payrolls", "FEDFUNDS": "fed_funds",
    "DGS2": "treasury_2y", "DGS10": "treasury_10y", "T10YIE": "breakeven_10y",
    "UMCSENT": "sentiment", "MICH": "inflation_expect",
    "VIXCLS": "vix", "DTWEXBGS": "dollar", "DCOILWTICO": "oil",
}

In [ ]:
cache = DATA / "macro_raw.csv"

if cache.exists():
    macro_raw = pd.read_csv(cache, index_col=0, parse_dates=True)
else:
    from pandas_datareader import data as web
    macro_raw = web.DataReader(list(SERIES), "fred", "2015-01-01")
    macro_raw.columns = [SERIES[c] for c in macro_raw.columns]
    macro_raw.to_csv(cache)

print(f"{macro_raw.shape[1]} series, {macro_raw.index.min():%Y-%m} to {macro_raw.index.max():%Y-%m}")

## 4. Align to monthly

Yields, VIX, the dollar and oil are daily. Fiserv stamps periods at month
start, so everything has to match — an off-by-one month here would be
invisible and would wreck everything downstream.

In [ ]:
macro = macro_raw.resample("MS").last()

assert (macro.index.day == 1).all() and (fiserv.index.day == 1).all()
print(f"{macro.shape[0]} months")

Series end in different months because of the release calendar — CPI and
retail sales for month *m* land mid *m+1*, PCE at the end of it. So PCE
always trails by a month.

In [ ]:
macro.apply(lambda s: s.last_valid_index()).value_counts().sort_index()

## 5. Missing values

Gaps inside a series are worth filling; gaps at the end are just data that
hasn't been published yet. `limit_area="inside"` keeps interpolation from
running past the last real observation.

In [ ]:
interior = macro.apply(lambda s: s.loc[s.first_valid_index():s.last_valid_index()].isna().sum())
print(interior[interior > 0] if interior.any() else "no interior gaps")

macro = macro.interpolate(limit=2, limit_area="inside")

## 6. Merge

Inner join on month start. Fiserv starts in 2019, so that binds — about
90 months, a small sample that should keep the feature count down later.

In [ ]:
df = fiserv.join(macro, how="inner")
print(f"{df.shape[0]} months x {df.shape[1]} columns, {df.index.min():%Y-%m} to {df.index.max():%Y-%m}")

## 7. Features

Log growth for levels, plain differences for rates — 2% to 3% is a
percentage-point move, and a log ratio breaks on negative values.

In [ ]:
RATES = ["fed_funds", "treasury_2y", "treasury_10y", "breakeven_10y",
         "savings_rate", "unemployment", "inflation_expect", "vix"]

levels = df.drop(columns=RATES)
rates = df[RATES]

growth = pd.concat([
    (np.log(levels).diff() * 100).add_suffix("_mom"),
    (np.log(levels).diff(12) * 100).add_suffix("_yoy"),
    rates.diff().add_suffix("_mom"),
    rates.diff(12).add_suffix("_yoy"),
], axis=1)

print(f"{growth.shape[1]} growth columns")

In [ ]:
smooth = ["fsbi_sales_sa_yoy", "fsbi_ticket_sa_yoy", "fsbi_txn_sa_yoy",
          "retail_sa_yoy", "cpi_sa_yoy"]

ma = pd.concat([growth[smooth].rolling(w).mean().add_suffix(f"_ma{w}")
                for w in (3, 6)], axis=1)
ma.tail(3).round(2)

### Lags

Lag *k* means predicting month *t* from data at *t−k*. Which lag is honest
depends on the calendar: Fiserv publishes month *m* on the 2nd of *m+1*,
CPI lands around the 13th. So for CPI, lag 0 is already public — a
**nowcast**. Lag 1 is a real one-month-ahead **forecast**.

Both are fine. Mixing them without saying so is how backtests flatter
themselves.

In [ ]:
signals = [c for c in growth.columns if c.startswith("fsbi_")]

lags = pd.concat([growth[signals].shift(k).add_suffix(f"_lag{k}")
                  for k in (1, 2, 3, 6, 12)], axis=1)

integrated = pd.concat([df, growth, ma, lags], axis=1)
print(f"{integrated.shape[1]} columns, {integrated.dropna().shape[0]} complete rows")

## 8. Scaling

Not applied to the saved file. Fitting a scaler on everything leaks
test-period means and standard deviations into training, so it has to be
refit inside each walk-forward fold.

In [ ]:
from sklearn.preprocessing import StandardScaler

X = integrated[["fsbi_ticket_sa_yoy", "fsbi_txn_sa_yoy"]].dropna()
train, test = X.iloc[:-12], X.iloc[-12:]

scaler = StandardScaler().fit(train)
print(f"train mean {scaler.transform(train).mean():+.2f}")
print(f"test mean  {scaler.transform(test).mean():+.2f}   <- not zero, which is correct")

## 9. Save

In [ ]:
integrated.to_csv(DATA / "processed/integrated_monthly.csv")

pd.DataFrame({
    "column": integrated.columns,
    "n_obs": integrated.notna().sum().values,
    "first": integrated.apply(lambda s: s.first_valid_index()).values,
}).to_csv(DATA / "processed/data_dictionary.csv", index=False)

print(f"saved {integrated.shape[0]} months x {integrated.shape[1]} columns")

## Notes for later phases

- Nominal, not real — the real index is deflated with CPI, so using it to
  predict CPI is circular.
- SA and NSA both kept. Nearly identical YoY, very different MoM. NSA is
  the right basis for anything traded, since CPI settles unadjusted.
- Saved unscaled; scale inside CV folds.
- Lag 0 is a nowcast, lag 1 is a forecast — keep that distinction in the
  backtest.
- ~90 months, mostly COVID and the inflation spike. Start with few features.